In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, subprocess, sys, time
import numpy as np
import pandas as pd
from scipy.special import gammainc, gammaincinv, gammaln
from scipy.integrate import quad

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
if importlib.util.find_spec('numpyro') is None:
    print("Installing NumPyro without changing Kaggle's JAX/CUDA packages...", flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps','numpyro==0.22.0'],check=True)
import jax
import jax.numpy as jnp
import jax.scipy as jsp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.infer.initialization import init_to_value
from numpyro.diagnostics import summary as numpyro_summary
import matplotlib.pyplot as plt

print('JAX backend:',jax.default_backend(),'devices:',jax.devices(),'NumPyro:',numpyro.__version__)
if jax.default_backend()!='gpu': raise RuntimeError('GPU preflight failed: select GPU T4 x2 and restart the session.')
def newest(pattern,label):
    found=list(INPUT.rglob(pattern))
    if not found: raise FileNotFoundError(f'Attach {label}: {pattern}')
    return max(found,key=lambda p:(p.stat().st_mtime,str(p)))

# Required inputs: 100K Day 7 synthetic binary simulations; Day 9.12 MLE seeds;
# Day 9.15 row-87483 duration profile; Day 9.16 baseline diagnostics.
binary_sources=[]
for mp in INPUT.rglob('simulation_metadata.json'):
    if mp.parent.name!='binary': continue
    try: meta=json.loads(mp.read_text(encoding='utf-8'))
    except Exception: continue
    if meta.get('family')=='binary' and int(meta.get('parameter_draws_unique',0))>=100_000:
        binary_sources.append((meta.get('created_utc',''),mp.parent))
if not binary_sources: raise FileNotFoundError('Attach the successful Day 7 Cosmic Swan Bayesian output with binary simulations.')
_,BINARY_DIR=sorted(binary_sources,key=lambda x:x[0])[-1]
OBS=np.load(BINARY_DIR/'observations.npy',mmap_mode='r')
PARAMETERS=pd.read_csv(BINARY_DIR/'parameters.csv')
TIME=np.asarray(np.load(BINARY_DIR/'time_days.npy'),dtype=float)
if OBS.ndim!=2 or OBS.shape!=(len(PARAMETERS),len(TIME)): raise ValueError('Day 7 arrays have incompatible shapes.')
MAP_PATH=newest('day9_12_conditional_mle_parameters.csv','Day 9.12 conditional MLE output')
MAP_DIAG_PATH=newest('day9_12_conditional_mle_diagnostics.csv','Day 9.12 MLE diagnostics')
PROFILE_PATH=newest('day9_15_nuisance_profile_summary.csv','Day 9.15 profile output')
BASELINE_PATH=newest('day9_16_case_diagnostics.csv','Day 9.16 baseline output')
MAP=pd.read_csv(MAP_PATH); MAP_DIAG=pd.read_csv(MAP_DIAG_PATH); PROFILES=pd.read_csv(PROFILE_PATH)
BASELINE=pd.read_csv(BASELINE_PATH)
ROWS=[87483,4923]
SEEDS={}
for row in ROWS:
    sub=MAP[MAP.simulation_row.astype(int).eq(row)]
    values=sub.set_index('parameter').estimate.astype(float).to_dict()
    need={'baseline_flux','phase','depth_primary','depth_ratio','duration_fraction','ellipsoidal_amplitude','reflection_amplitude','noise_sigma'}
    if need-set(values): raise ValueError(f'Missing MLE seed terms for {row}: {sorted(need-set(values))}')
    diag=MAP_DIAG[MAP_DIAG.simulation_row.astype(int).eq(row)]
    if len(diag)!=1 or not bool(diag.iloc[0].optimizer_success): raise ValueError(f'Day 9.12 MLE did not converge for row {row}.')
    values['period_fixed_days']=float(diag.iloc[0].period_fixed_days)
    SEEDS[row]=values
p874=PROFILES[PROFILES.simulation_row.astype(int).eq(87483)&PROFILES.profile_parameter.eq('duration_fraction')]
if len(p874)!=1: raise ValueError('Day 9.15 duration profile for row 87483 is missing/duplicated.')
DURATION_FIXED=float(p874.iloc[0].profile_minimum)
if not .005<=DURATION_FIXED<=.12: raise ValueError('Profile duration outside model support.')
if max(ROWS)>=len(PARAMETERS): raise ValueError('Selected row outside Day 7 simulation table.')

# Original Day 9.16 priors were Uniform(log(1e-4), log(.03)) on log(sigma).
# Integrate that same log-uniform noise prior exactly, reducing the HMC dimension by one.
SIGMA_MIN,SIGMA_MAX=1e-4,.03
Z_LO,Z_HI=np.log(SIGMA_MIN),np.log(SIGMA_MAX)
OUT=WORK/('day9-19-marginalized-noise-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True,exist_ok=False)

def smooth_clip_np(x,k=500.): return (np.logaddexp(0,k*x)-np.logaddexp(0,k*(x-1.)))/k
def predict_np(t,period,phase,depth,ratio,duration,ellip,reflection,baseline):
    orb=np.mod(t/period-phase+.5,1.)-.5
    sec=np.mod(orb,1.)-.5
    half=max(duration/2.,1e-6); ingress=max(half*.18,1e-6)
    primary=smooth_clip_np((half+ingress-np.abs(orb))/(2*ingress))
    secondary=smooth_clip_np((half+ingress-np.abs(sec))/(2*ingress))
    return baseline-depth*primary-depth*ratio*secondary+ellip*np.cos(4*np.pi*orb)+reflection*np.cos(2*np.pi*orb)

def numpy_log_marginal(sse,n):
    alpha=n/2.; beta=max(sse/2.,1e-300)
    p_hi=gammainc(alpha,beta/SIGMA_MIN**2)
    p_lo=gammainc(alpha,beta/SIGMA_MAX**2)
    diff=max(p_hi-p_lo,1e-300)
    return -.5*n*np.log(2*np.pi)-np.log(Z_HI-Z_LO)-np.log(2.)+gammaln(alpha)-alpha*np.log(beta)+np.log(diff)

def rows_data(row):
    t=TIME.copy(); y=np.asarray(OBS[row],dtype=float)
    keep=np.isfinite(t)&np.isfinite(y); t,y=t[keep],y[keep]
    order=np.argsort(t); t,y=t[order],y[order]; t=t-t.min()
    return t,y

# Validate the closed-form integral against direct numerical quadrature at each MLE seed.
for row in ROWS:
    t,y=rows_data(row); s=SEEDS[row]; duration=DURATION_FIXED if row==87483 else s['duration_fraction']
    pred=predict_np(t,s['period_fixed_days'],s['phase']%1.,s['depth_primary'],s['depth_ratio'],duration,
        s['ellipsoidal_amplitude'],s['reflection_amplitude'],s['baseline_flux'])
    sse=float(np.sum((y-pred)**2)); n=len(y); alpha=n/2.; beta=sse/2.
    zstar=float(np.clip(.5*np.log(max(sse/n,1e-300)),Z_LO,Z_HI))
    def log_integrand(z): return -.5*n*np.log(2*np.pi)-n*z-sse/(2*np.exp(2*z))-np.log(Z_HI-Z_LO)
    peak=log_integrand(zstar)
    q=quad(lambda z: np.exp(log_integrand(z)-peak),Z_LO,Z_HI,epsabs=1e-12,epsrel=1e-10,limit=200)[0]
    numeric=peak+np.log(q)
    analytic=numpy_log_marginal(sse,n)
    delta=abs(numeric-analytic)
    if not np.isfinite(delta) or delta>1e-5: raise RuntimeError(f'Noise integral check failed row {row}: log-density difference={delta}')
    print(f'[PASS] exact noise marginalization matches quadrature for row {row}; abs log-density error={delta:.2e}')
print('[PASS] Inputs validated. No sampler has started yet.')

case_rows=[]; parameter_rows=[]; interval_rows=[]; plot_rows=[]
for row in ROWS:
    started=time.time(); seed=SEEDS[row]; period=float(seed['period_fixed_days'])
    duration_fixed=(row==87483); duration_const=DURATION_FIXED if duration_fixed else None
    warmup,samples=(500,800) if duration_fixed else (500,1200)
    t,y=rows_data(row); n=len(y); alpha=n/2.
    phase0=float(seed['phase']%1.); time_j,obs_j=jnp.asarray(t),jnp.asarray(y)
    def smooth_clip_jax(x,k=500.): return (jax.nn.softplus(k*x)-jax.nn.softplus(k*(x-1.)))/k
    def predict_jax(period,phase,depth,ratio,duration,ellip,reflection,baseline):
        orb=jnp.mod(time_j/period-phase+.5,1.)-.5
        sec=jnp.mod(orb,1.)-.5
        half=jnp.maximum(duration/2.,1e-6); ingress=jnp.maximum(half*.18,1e-6)
        primary=smooth_clip_jax((half+ingress-jnp.abs(orb))/(2*ingress))
        secondary=smooth_clip_jax((half+ingress-jnp.abs(sec))/(2*ingress))
        return baseline-depth*primary-depth*ratio*secondary+ellip*jnp.cos(4*jnp.pi*orb)+reflection*jnp.cos(2*jnp.pi*orb)
    def model():
        baseline=numpyro.sample('baseline_flux',dist.Normal(1.,.02))
        phase_offset=numpyro.sample('phase_offset',dist.Normal(0.,.025))
        depth=numpyro.sample('depth_primary',dist.Uniform(.002,.45))
        ratio=numpyro.sample('depth_ratio',dist.Uniform(0.,1.))
        duration=jnp.asarray(duration_const) if duration_fixed else numpyro.sample('duration_fraction',dist.Uniform(.005,.12))
        ellip=numpyro.sample('ellipsoidal_amplitude',dist.Uniform(0.,.02))
        reflection=numpyro.sample('reflection_amplitude',dist.Uniform(-.01,.01))
        phase=jnp.mod(phase0+phase_offset,1.)
        pred=predict_jax(period,phase,depth,ratio,duration,ellip,reflection,baseline)
        sse=jnp.maximum(jnp.sum((obs_j-pred)**2),1e-30)
        beta=sse/2.; a=jnp.asarray(n/2.)
        p_hi=jsp.special.gammainc(a,beta/(SIGMA_MIN**2))
        p_lo=jsp.special.gammainc(a,beta/(SIGMA_MAX**2))
        p_diff=jnp.maximum(p_hi-p_lo,1e-30)
        log_marginal=(-.5*n*jnp.log(2*jnp.pi)-jnp.log(Z_HI-Z_LO)-jnp.log(2.)+
            jsp.special.gammaln(a)-a*jnp.log(beta)+jnp.log(p_diff))
        numpyro.factor('exact_log_uniform_noise_marginal_likelihood',log_marginal)
    init={'baseline_flux':float(np.clip(seed['baseline_flux'],.9501,1.0499)),'phase_offset':0.,
        'depth_primary':float(np.clip(seed['depth_primary'],.002001,.449999)),
        'depth_ratio':float(np.clip(seed['depth_ratio'],1e-5,.99999)),
        'ellipsoidal_amplitude':float(np.clip(seed['ellipsoidal_amplitude'],1e-5,.01999)),
        'reflection_amplitude':float(np.clip(seed['reflection_amplitude'],-.00999,.00999))}
    if not duration_fixed: init['duration_fraction']=float(np.clip(seed['duration_fraction'],.005001,.119999))
    latent=tuple(init)
    kernel=NUTS(model,init_strategy=init_to_value(values=init),target_accept_prob=.90,max_tree_depth=10,dense_mass=[latent])
    mcmc=MCMC(kernel,num_warmup=warmup,num_samples=samples,num_chains=2,
        chain_method='parallel' if len(jax.devices())>=2 else 'sequential',progress_bar=True)
    try:
        mcmc.run(jax.random.PRNGKey(190000+row),extra_fields=('diverging','accept_prob','num_steps'))
        grouped={k:np.asarray(v) for k,v in mcmc.get_samples(group_by_chain=True).items()}
        derived={k:v.copy() for k,v in grouped.items()}
        derived['phase']=(phase0+derived['phase_offset'])%1.
        if duration_fixed: derived['duration_fraction']=np.full((2,samples),duration_const)
        # Exact conditional draw: u=beta/sigma^2 follows Gamma(n/2,1), truncated by sigma bounds.
        d=derived; phase=d['phase'].reshape(-1,1); dep=d['depth_primary'].reshape(-1,1)
        ratio=d['depth_ratio'].reshape(-1,1); dur=d['duration_fraction'].reshape(-1,1)
        ell=d['ellipsoidal_amplitude'].reshape(-1,1); refl=d['reflection_amplitude'].reshape(-1,1)
        base=d['baseline_flux'].reshape(-1,1)
        orb=np.mod(t.reshape(1,-1)/period-phase+.5,1.)-.5; sec=np.mod(orb,1.)-.5
        half=np.maximum(dur/2.,1e-6); ingress=np.maximum(half*.18,1e-6)
        prim=smooth_clip_np((half+ingress-np.abs(orb))/(2*ingress))
        seco=smooth_clip_np((half+ingress-np.abs(sec))/(2*ingress))
        pred=base-dep*prim-dep*ratio*seco+ell*np.cos(4*np.pi*orb)+refl*np.cos(2*np.pi*orb)
        sse=np.maximum(np.sum((y.reshape(1,-1)-pred)**2,axis=1),1e-30); beta=sse/2.
        p_lo=gammainc(alpha,beta/SIGMA_MAX**2); p_hi=gammainc(alpha,beta/SIGMA_MIN**2)
        if np.any(~np.isfinite(p_hi-p_lo)) or np.any((p_hi-p_lo)<1e-12):
            raise RuntimeError('Conditional noise CDF interval is numerically unresolved.')
        rng=np.random.default_rng(191000+row)
        u_prob=p_lo+rng.random(len(beta))*(p_hi-p_lo)
        u=np.maximum(gammaincinv(alpha,np.clip(u_prob,1e-14,1-1e-14)),1e-300)
        sigma=np.sqrt(beta/u).reshape(2,samples)
        derived['noise_sigma']=sigma; derived['log_noise']=np.log(sigma)
        diag_input={k:v for k,v in grouped.items()}
        diag_input['log_noise']=derived['log_noise']
        diag=numpyro_summary(diag_input,group_by_chain=True)
        per=[]
        for name,item in diag.items():
            rh=float(np.nanmax(np.asarray(item.get('r_hat',np.nan))))
            es=float(np.nanmin(np.asarray(item.get('n_eff',np.nan))))
            per.append((name,rh,es))
            parameter_rows.append({'simulation_row':row,'parameter':name,'r_hat':rh,'effective_sample_size':es})
        maxr=max(x[1] for x in per); mine=min(x[2] for x in per)
        extras=mcmc.get_extra_fields(group_by_chain=True)
        div=int(np.asarray(extras['diverging']).sum()); steps=np.asarray(extras['num_steps'],float)
        frac_depth=float(np.mean(steps >= (2**10-1)))
        status='screen_pass' if maxr<=1.05 and mine>=100 and div==0 and frac_depth<=.05 else 'needs_review'
        truth=PARAMETERS.iloc[row]
        truth_map={'baseline_flux':1.,'phase_offset':(float(truth.phase)-phase0+.5)%1.-.5,
            'depth_primary':float(truth.depth_primary),'depth_ratio':float(truth.depth_ratio),
            'duration_fraction':float(truth.duration_fraction),'ellipsoidal_amplitude':float(truth.ellipsoidal_amplitude),
            'reflection_amplitude':float(truth.reflection_amplitude),'log_noise':float(np.log(truth.noise_sigma)),
            'noise_sigma':float(truth.noise_sigma)}
        for name,arr in derived.items():
            if name=='duration_fraction' and duration_fixed: continue
            if name not in truth_map: continue
            q05,med,q95=np.quantile(arr,[.05,.5,.95])
            interval_rows.append({'simulation_row':row,'parameter':name,'period_fixed_days':period,
                'duration_fixed':duration_fixed,'duration_fixed_value':duration_const,
                'q05':float(q05),'median':float(med),'q95':float(q95),
                'known_truth_evaluation_only':truth_map[name],
                'truth_inside_90pct_interval':bool(q05<=truth_map[name]<=q95),
                'scope':'conditional_on_profile_period_and_duration' if duration_fixed else 'conditional_on_profile_period'})
        arrays={k:np.asarray(v) for k,v in derived.items()}
        np.savez_compressed(OUT/f'row_{row}_posterior_by_chain.npz',**arrays)
        med={k:float(np.median(v)) for k,v in derived.items()}
        pred_med=predict_np(t,period,med['phase'],med['depth_primary'],med['depth_ratio'],
            duration_const if duration_fixed else med['duration_fraction'],med['ellipsoidal_amplitude'],
            med['reflection_amplitude'],med['baseline_flux'])
        plot_rows.append((row,t,y,pred_med,period,duration_const))
        result={'simulation_row':row,'status':status,'period_fixed_days':period,'duration_fixed':duration_fixed,
            'duration_fixed_value':duration_const,'noise_prior':'Uniform(log_sigma_min, log_sigma_max), analytically marginalized',
            'chains':2,'warmup_per_chain':warmup,'samples_per_chain':samples,'max_r_hat':maxr,
            'min_effective_sample_size':mine,'divergences':div,'fraction_at_max_tree_depth':frac_depth,
            'mean_accept_probability':float(np.mean(np.asarray(extras['accept_prob']))),
            'median_leapfrog_steps':float(np.median(steps)),'seconds':round(time.time()-started,1)}
        case_rows.append(result)
    except Exception as exc:
        result={'simulation_row':row,'status':'failed','period_fixed_days':period,
            'duration_fixed':duration_fixed,'duration_fixed_value':duration_const,
            'error':f'{type(exc).__name__}: {exc}','seconds':round(time.time()-started,1)}
        case_rows.append(result)
    pd.DataFrame(case_rows).to_csv(OUT/'day9_19_case_diagnostics.csv',index=False)
    pd.DataFrame(parameter_rows).to_csv(OUT/'day9_19_per_parameter_diagnostics.csv',index=False)
    pd.DataFrame(interval_rows).to_csv(OUT/'day9_19_posterior_intervals.csv',index=False)
    print('Marginal-noise NUTS result:',result,flush=True)

if plot_rows:
    fig,axes=plt.subplots(1,len(plot_rows),figsize=(14,5),squeeze=False,constrained_layout=True)
    for ax,(row,t,y,pred,period,dur) in zip(axes.flat,plot_rows):
        ax.scatter(t,y,s=5,alpha=.25,label='synthetic data'); ax.plot(t,pred,color='crimson',lw=1,label='posterior median')
        ax.set_title(f'Row {row} | period fixed {period:.6f} d'+(f' | duration fixed {dur:.6f}' if dur is not None else ''))
        ax.set_xlabel('Time (days)'); ax.set_ylabel('Normalized flux'); ax.legend(fontsize=8)
    fig.savefig(OUT/'day9_19_marginal_noise_fits.png',dpi=140); plt.show()
case_df=pd.DataFrame(case_rows); param_df=pd.DataFrame(parameter_rows); interval_df=pd.DataFrame(interval_rows)
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'day7_binary_input':str(BINARY_DIR),
    'day9_12_mle_input':str(MAP_PATH.parent),'day9_15_profile_input':str(PROFILE_PATH.parent),
    'day9_16_baseline_input':str(BASELINE_PATH.parent),'cases':case_df.to_dict(orient='records'),
    'method':'Exact integration over the same bounded log-uniform Gaussian noise prior used in Day 9.16; posterior noise is then sampled from its exact conditional inverse-gamma form.',
    'interpretation_limits':['Period is fixed at the Day 9.12 MLE mode for both cases.','Row 87483 also fixes duration at the Day 9.15 profile optimum; no duration posterior interval is produced.',
        'This is a conditional synthetic recovery experiment, not calibration or real-star validation.','A successful sampler screen is necessary but does not establish model validity.'],
    'outputs':['day9_19_case_diagnostics.csv','day9_19_per_parameter_diagnostics.csv','day9_19_posterior_intervals.csv',
        'day9_19_marginal_noise_fits.png','row_*_posterior_by_chain.npz']}
(OUT/'day9_19_marginal_noise_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('\nCase summary:\n',case_df.to_string(index=False))
print('\nPer-parameter diagnostics:\n',param_df.to_string(index=False))
print('\nSaved outputs:',OUT)
print(json.dumps(summary,indent=2))